# Predict Bell-pair measurement parity

Prepare a two-qubit Bell pair. Your seed selects a phase flip on qubit 0 and a bit flip on qubit 1. Before running each cell, predict which two outcomes will occur in the Z basis and which two will occur in the X basis. A phase flip changes the X-basis parity. A bit flip changes the Z-basis parity.

This notebook uses an ideal local Qiskit statevector. Compare the two exact distributions before examining noisy hardware measurements in a separate exercise.

In a fresh Colab session, install Qiskit in a new cell before running this exercise:

```python
%pip install qiskit==2.6.2
```

Local Jupyter users can install the same version in their own environment. This exercise uses synthetic input and sends no results to Skills Lab.

In [ ]:
from hashlib import sha256
import json
import re
from pathlib import Path
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

EXERCISE = "qiskit-bell/v1"
seed = "00112233445566778899aabbccddeeff"
if not re.fullmatch(r"[0-9a-f]{32}", seed):
    raise ValueError("Enter exactly 32 lowercase hexadecimal characters.")

selector = int(sha256(f"skills-lab/{EXERCISE}:{seed}".encode()).hexdigest()[0], 16)
phase_flip = bool(selector & 1)
bit_flip = bool(selector & 2)

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
if phase_flip:
    circuit.z(0)
if bit_flip:
    circuit.x(1)
print("Phase flip:", phase_flip, "Bit flip:", bit_flip)


In [ ]:
def half_units(operations):
    probabilities = Statevector.from_instruction(operations).probabilities()
    output = [round(float(probability) * 2) for probability in probabilities]
    if sum(output) != 2 or any(abs(float(p) * 2 - n) > 1e-9 for p, n in zip(probabilities, output)):
        raise ValueError("The circuit must produce exact multiples of one half.")
    return output

z_units = half_units(circuit)
x_circuit = circuit.copy()
x_circuit.h(0)
x_circuit.h(1)
x_units = half_units(x_circuit)
for label, z, x in zip(("00", "01", "10", "11"), z_units, x_units):
    print(label, f"Z {z}/2", f"X {x}/2")


Explain which operation changes each distribution. In the displayed bitstrings, qubit 1 is the left bit and qubit 0 is the right bit. Applying a Hadamard to both qubits before the final computational-basis readout gives the X-basis distribution.

In [ ]:
path = Path("qiskit-bell-result.json")
path.write_text(json.dumps([EXERCISE, seed, z_units, x_units]) + "\n", encoding="utf-8")
print(path.name)
